In [1]:
import platform, os
print(platform.system())
print(round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1), "GB RAM")

Linux
13.6 GB RAM


In [4]:
! pip install -q --upgrade kaggle duckdb implicit

In [2]:
import os, getpass
from pathlib import Path

token = getpass.getpass("Paste your Kaggle token: ").strip()
print("token length:", len(token))

os.environ["KAGGLE_API_TOKEN"] = token
kdir = Path.home() / ".kaggle"
kdir.mkdir(exist_ok=True)
(kdir / "access_token").write_text(token)
os.chmod(kdir / "access_token", 0o600)

token length: 37


In [5]:
COMP = "h-and-m-personalized-fashion-recommendations"
for f in ["transactions_train.csv", "customers.csv", "articles.csv"]:
    !kaggle competitions download -c {COMP} -f {f} -p /content/raw -q
! cd /content/raw && unzip -o -q '*.zip' && rm -f *.zip
! ls -lh /content/raw


3 archives were successfully processed.
total 3.5G
-rw-r--r-- 1 root root  35M Jan 17  2022 articles.csv
-rw-r--r-- 1 root root 198M Jan 17  2022 customers.csv
-rw-r--r-- 1 root root 3.3G Jan 17  2022 transactions_train.csv


In [6]:
!rm -rf /content/repo
!git clone -q https://github.com/WoahImCoding/ecommerce-recommender.git /content/repo
!ls /content/repo/src

import sys
sys.path.append("/content/repo")
from src.metrics import evaluate
print("metrics imported")

metrics.py
metrics imported


Main Table creation

In [11]:
import duckdb
from pathlib import Path

raw, proc = Path("/content/raw"), Path("/content/processed")
proc.mkdir(exist_ok=True)

con = duckdb.connect()
for name in ["transactions_train", "customers", "articles"]:
    con.execute(f"COPY (SELECT * FROM read_csv_auto('{raw}/{name}.csv')) TO '{proc}/{name}.parquet' (FORMAT PARQUET)")

con.execute(f"CREATE VIEW transactions AS SELECT * FROM '{proc}/transactions_train.parquet'")
con.execute(f"CREATE VIEW articles AS SELECT * FROM '{proc}/articles.parquet'")
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *, CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
con.sql("SELECT count(*) AS n FROM tx").df()

,n
0,31788324


In [13]:
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *,
           CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")

con.sql("""
    SELECT week_ago,
           min(t_dat)                  AS first_day,
           max(t_dat)                  AS last_day,
           count(*)                    AS n_tx,
           count(DISTINCT customer_id) AS n_customers
    FROM tx
    WHERE week_ago <= 4
    GROUP BY 1
    ORDER BY 1
""").df()

,week_ago,first_day,last_day,n_tx,n_customers
0,0,2020-09-16,2020-09-22,240311,68984
1,1,2020-09-09,2020-09-15,255241,72019
2,2,2020-09-02,2020-09-08,264657,75822
3,3,2020-08-26,2020-09-01,283181,80253
4,4,2020-08-19,2020-08-25,256644,72035


In [14]:
# Give every customer a small integer ID
con.execute("""
    CREATE OR REPLACE TABLE cust_map AS
    SELECT customer_id,
           CAST(row_number() OVER (ORDER BY customer_id) - 1 AS INTEGER) AS cust
    FROM (SELECT DISTINCT customer_id FROM tx)
""")

# Load only the last 16 weeks, with compact column types
recent = con.sql("""
    SELECT m.cust,
           CAST(t.article_id AS INTEGER)       AS article_id,
           t.t_dat,
           CAST(t.week_ago AS SMALLINT)        AS week_ago,
           CAST(t.price AS FLOAT)              AS price,
           CAST(t.sales_channel_id AS TINYINT) AS channel
    FROM tx t
    JOIN cust_map m USING (customer_id)
    WHERE t.week_ago <= 15
""").df()

print(f"{len(recent):,} rows, {recent.memory_usage(deep=True).sum() / 1e6:.0f} MB")
recent.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5,063,720 rows, 116 MB


,cust,article_id,t_dat,week_ago,price,channel
0,321738,881823004,2020-06-04,15,0.038966,2
1,321738,851010006,2020-06-04,15,0.015576,2
2,321738,825600002,2020-06-04,15,0.023373,2
3,321767,373506004,2020-06-04,15,0.013542,1
4,321767,854678003,2020-06-04,15,0.008458,1


In [15]:
labels = con.sql("""
    SELECT customer_id, list(DISTINCT article_id) AS items
    FROM tx WHERE week_ago = 0 GROUP BY 1
""").df()
actual = dict(zip(labels["customer_id"], labels["items"].apply(set)))

recent_top = con.sql("""
    SELECT article_id FROM tx WHERE week_ago = 1
    GROUP BY 1 ORDER BY count(*) DESC LIMIT 12
""").df()["article_id"].tolist()

print(len(actual), evaluate(lambda c: recent_top, actual))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

68984 {'recall@12': np.float64(0.0255), 'map@12': np.float64(0.0087)}


Test - Train Split

In [16]:
import numpy as np
import pandas as pd

test = recent[recent["week_ago"] == 0]                  # the week we predict
hist = recent[recent["week_ago"].between(1, 12)]        # the 12 weeks before it

actual = test.groupby("cust")["article_id"].apply(set).to_dict()
print(len(actual), "test customers")                    # should be 68984

68984 test customers


In [17]:
rep = (hist.groupby(["cust", "article_id"])
           .agg(last_day=("t_dat", "max"), n_bought=("t_dat", "size"))
           .reset_index()
           .sort_values(["cust", "last_day", "n_bought"], ascending=[True, False, False]))
rep["rank"] = rep.groupby("cust").cumcount()
rep = rep[rep["rank"] < 30]

rep_lists = rep.groupby("cust")["article_id"].apply(list).to_dict()
print(len(rep_lists), "customers have repurchase candidates")

505310 customers have repurchase candidates


In [18]:
trend = hist.loc[hist["week_ago"] == 1, "article_id"].value_counts().index[:30].tolist()

Measure candidate recall

In [19]:
def candidate_recall(get_cands, actual):
    """Share of each customer's purchases that appear anywhere in their candidate list."""
    vals = [len(items & set(get_cands(c))) / len(items) for c, items in actual.items()]
    return round(float(np.mean(vals)), 4)

def union_cands(c):
    return set(rep_lists.get(c, [])) | set(trend)

pd.DataFrame({
    "candidate_recall": [
        candidate_recall(lambda c: rep_lists.get(c, []), actual),
        candidate_recall(lambda c: trend, actual),
        candidate_recall(union_cands, actual),
    ],
    "avg_candidates": [
        np.mean([len(rep_lists.get(c, [])) for c in actual]),
        len(trend),
        np.mean([len(union_cands(c)) for c in actual]),
    ],
}, index=["Repurchase (up to 30)", "Trending (30)", "Both combined"]).round(4)

,candidate_recall,avg_candidates
Repurchase (up to 30),0.0378,6.6337
Trending (30),0.0511,30.0000
Both combined,0.0862,36.5205


Build the Customer x item matrix

In [20]:
import scipy.sparse as sp

hist = hist.copy()
hist["t_dat"] = pd.to_datetime(hist["t_dat"])

users = hist["cust"].unique()
items = hist["article_id"].unique()
u_idx = pd.Series(np.arange(len(users)), index=users)   # customer ID -> matrix row
i_idx = pd.Series(np.arange(len(items)), index=items)   # article ID  -> matrix column

# Recent purchases count more: weight halves every 28 days
age_days = (hist["t_dat"].max() - hist["t_dat"]).dt.days.to_numpy()
weight = (0.5 ** (age_days / 28)).astype(np.float32)

mat = sp.csr_matrix(
    (weight, (u_idx[hist["cust"]].to_numpy(), i_idx[hist["article_id"]].to_numpy())),
    shape=(len(users), len(items)),
)
print(mat.shape, f"{mat.nnz:,} non-zero cells")

(505310, 41376) 3,237,697 non-zero cells


Train ALS: Alternating Least Squares

In [21]:
from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.05, random_state=42)
als.fit(mat)

/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Top 30 ALS Candidates for each test customer

In [22]:
test_users = [c for c in actual if c in u_idx.index]       # ALS only knows customers with history
rows = u_idx[test_users].to_numpy()
ids, scores = als.recommend(rows, mat[rows], N=30, filter_already_liked_items=False)

als_lists = {c: items[ids[k]].tolist() for k, c in enumerate(test_users)}
print(len(als_lists), "customers have ALS candidates")

49219 customers have ALS candidates


ALS to the candidate recall table

In [23]:
def union3(c):
    return set(rep_lists.get(c, [])) | set(trend) | set(als_lists.get(c, []))

pd.DataFrame({
    "candidate_recall": [
        candidate_recall(lambda c: rep_lists.get(c, []), actual),
        candidate_recall(lambda c: trend, actual),
        candidate_recall(lambda c: als_lists.get(c, []), actual),
        candidate_recall(union_cands, actual),
        candidate_recall(union3, actual),
    ],
    "avg_candidates": [
        np.mean([len(rep_lists.get(c, [])) for c in actual]),
        len(trend),
        np.mean([len(als_lists.get(c, [])) for c in actual]),
        np.mean([len(union_cands(c)) for c in actual]),
        np.mean([len(union3(c)) for c in actual]),
    ],
}, index=["Repurchase", "Trending", "ALS", "Repurchase + trending", "All three"]).round(4)

,candidate_recall,avg_candidates
Repurchase,0.0378,6.6337
Trending,0.0511,30.0000
ALS,0.0375,21.4045
Repurchase + trending,0.0862,36.5205
All three,0.1060,55.3455


Items bought together: recommend that to the customers already purchased one item from the bucket

In [24]:
baskets = hist.loc[hist["week_ago"] <= 4, ["cust", "t_dat", "article_id"]].drop_duplicates()
con.register("b", baskets)

cop = con.sql("""
    WITH pairs AS (
        SELECT x.article_id AS src, y.article_id AS dst, count(*) AS n
        FROM b x
        JOIN b y
          ON x.cust = y.cust AND x.t_dat = y.t_dat AND x.article_id <> y.article_id
        GROUP BY 1, 2
    ),
    src_total AS (
        SELECT article_id AS src, count(*) AS n_src FROM b GROUP BY 1
    )
    SELECT src, dst, CAST(n AS DOUBLE) / n_src AS p
    FROM pairs JOIN src_total USING (src)
    QUALIFY row_number() OVER (PARTITION BY src ORDER BY n DESC, dst) <= 10
""").df()

print(f"{len(cop):,} item pairs")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

235,832 item pairs


Pairs into Candidates for Each customers

In [25]:
last5 = (hist.sort_values("t_dat", ascending=False)
             .drop_duplicates(["cust", "article_id"])
             .groupby("cust").head(5)[["cust", "article_id"]])
last5 = last5[last5["cust"].isin(list(actual))]

m = last5.merge(cop, left_on="article_id", right_on="src")
s = (m.groupby(["cust", "dst"])["p"].sum()
      .reset_index()
      .sort_values(["cust", "p"], ascending=[True, False]))
cop_lists = s.groupby("cust").head(20).groupby("cust")["dst"].apply(list).to_dict()
print(len(cop_lists), "customers have co-purchase candidates")

49122 customers have co-purchase candidates


In [26]:
week1 = hist.loc[hist["week_ago"] == 1, "article_id"].value_counts().index
trend60, trend100 = week1[:60].tolist(), week1[:100].tolist()

def union_with(extra_trend, use_cop):
    def f(c):
        s = set(rep_lists.get(c, [])) | set(extra_trend) | set(als_lists.get(c, []))
        if use_cop:
            s |= set(cop_lists.get(c, []))
        return s
    return f

variants = {
    "All three (trending 30)":           union_with(trend, False),
    "+ co-purchase (trending 30)":       union_with(trend, True),
    "+ co-purchase, trending 60":        union_with(trend60, True),
    "+ co-purchase, trending 100":       union_with(trend100, True),
}
pd.DataFrame({
    "candidate_recall": [candidate_recall(f, actual) for f in variants.values()],
    "avg_candidates":   [np.mean([len(f(c)) for c in actual]) for f in variants.values()],
}, index=variants.keys()).round(4)

,candidate_recall,avg_candidates
All three (trending 30),0.1060,55.3455
+ co-purchase (trending 30),0.1196,65.7290
"+ co-purchase, trending 60",0.1450,93.9273
"+ co-purchase, trending 100",0.1755,131.3563


Function That builds candidates for any week

In [27]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

def build_candidates(recent, label_week, n_trend=100, lookback=12, n_rep=30, n_als=30, n_cop=20):
    """Candidates (+ labels) for one week. History = the `lookback` weeks before label_week."""
    hist = recent[recent["week_ago"].between(label_week + 1, label_week + lookback)].copy()
    hist["t_dat"] = pd.to_datetime(hist["t_dat"])
    labels = recent.loc[recent["week_ago"] == label_week, ["cust", "article_id"]].drop_duplicates()
    custs = labels["cust"].unique()
    h = hist[hist["cust"].isin(custs)]

    # 1. Repurchase
    rep = (h.groupby(["cust", "article_id"])
             .agg(last_day=("t_dat", "max"), rep_count=("t_dat", "size"))
             .reset_index()
             .sort_values(["cust", "last_day", "rep_count"], ascending=[True, False, False]))
    rep["rep_rank"] = rep.groupby("cust").cumcount()
    rep = rep.loc[rep["rep_rank"] < n_rep, ["cust", "article_id", "rep_rank", "rep_count"]]

    # 2. Trending (same list for everyone)
    top = hist.loc[hist["week_ago"] == label_week + 1, "article_id"].value_counts().index[:n_trend].to_numpy()
    trend = pd.DataFrame({"cust": np.repeat(custs, len(top)),
                          "article_id": np.tile(top, len(custs)),
                          "trend_rank": np.tile(np.arange(len(top)), len(custs))})

    # 3. ALS
    users, items = hist["cust"].unique(), hist["article_id"].unique()
    u_idx = pd.Series(np.arange(len(users)), index=users)
    i_idx = pd.Series(np.arange(len(items)), index=items)
    age = (hist["t_dat"].max() - hist["t_dat"]).dt.days.to_numpy()
    w = (0.5 ** (age / 28)).astype(np.float32)
    mat = sp.csr_matrix((w, (u_idx[hist["cust"]].to_numpy(), i_idx[hist["article_id"]].to_numpy())),
                        shape=(len(users), len(items)))
    als = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.05, random_state=42)
    als.fit(mat, show_progress=False)
    warm = custs[np.isin(custs, users)]
    rows = u_idx[warm].to_numpy()
    ids, scores = als.recommend(rows, mat[rows], N=n_als, filter_already_liked_items=False)
    als_df = pd.DataFrame({"cust": np.repeat(warm, ids.shape[1]),
                           "article_id": items[ids.ravel()],
                           "als_rank": np.tile(np.arange(ids.shape[1]), len(warm)),
                           "als_score": scores.ravel()})

    # 4. Co-purchase
    baskets = hist.loc[hist["week_ago"] <= label_week + 4, ["cust", "t_dat", "article_id"]].drop_duplicates()
    con.register("b", baskets)
    pairs = con.sql("""
        WITH pairs AS (
            SELECT x.article_id AS src, y.article_id AS dst, count(*) AS n
            FROM b x JOIN b y
              ON x.cust = y.cust AND x.t_dat = y.t_dat AND x.article_id <> y.article_id
            GROUP BY 1, 2
        ),
        src_total AS (SELECT article_id AS src, count(*) AS n_src FROM b GROUP BY 1)
        SELECT src, dst, CAST(n AS DOUBLE) / n_src AS p
        FROM pairs JOIN src_total USING (src)
        QUALIFY row_number() OVER (PARTITION BY src ORDER BY n DESC, dst) <= 10
    """).df()
    last5 = (h.sort_values("t_dat", ascending=False)
               .drop_duplicates(["cust", "article_id"])
               .groupby("cust").head(5)[["cust", "article_id"]])
    cop = (last5.merge(pairs, left_on="article_id", right_on="src")
                .groupby(["cust", "dst"])["p"].sum().rename("cop_score").reset_index()
                .rename(columns={"dst": "article_id"})
                .sort_values(["cust", "cop_score"], ascending=[True, False]))
    cop = cop.groupby("cust").head(n_cop)

    # Combine: one row per (customer, candidate item), with every source's columns
    cands = pd.concat([d[["cust", "article_id"]] for d in (rep, trend, als_df, cop)]).drop_duplicates()
    for d in (rep, trend, als_df, cop):
        cands = cands.merge(d, on=["cust", "article_id"], how="left")
    for name, col in (("rep", "rep_rank"), ("trend", "trend_rank"), ("als", "als_rank"), ("cop", "cop_score")):
        cands[f"src_{name}"] = cands[col].notna().astype("int8")

    # Label: 1 if the customer bought this item in the label week
    cands = cands.merge(labels.assign(label=1), on=["cust", "article_id"], how="left")
    cands["label"] = cands["label"].fillna(0).astype("int8")
    cands["week"] = np.int8(label_week)
    floats = cands.select_dtypes("float64").columns
    cands[floats] = cands[floats].astype("float32")          # halve memory
    return cands, hist

In [28]:
cands0, hist0 = build_candidates(recent, label_week=0)

pos = cands0.groupby("cust")["label"].sum()
n_actual = pd.Series({c: len(a) for c, a in actual.items()})
print("candidate recall:", round(float((pos.reindex(n_actual.index, fill_value=0) / n_actual).mean()), 4))
print(f"{len(cands0):,} rows, {cands0['cust'].nunique():,} customers, "
      f"{len(cands0) / cands0['cust'].nunique():.0f} per customer, "
      f"{cands0.memory_usage(deep=True).sum() / 1e6:.0f} MB")
cands0.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

candidate recall: 0.1757
9,061,809 rows, 68,984 customers, 131 per customer, 344 MB


,cust,article_id,rep_rank,rep_count,trend_rank,als_rank,als_score,cop_score,src_rep,src_trend,src_als,src_cop,label,week
0,86,889036004,0.0,1.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0
1,86,880017001,1.0,2.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0
2,86,621381012,2.0,1.0,NaN,NaN,NaN,0.157803,1,0,0,1,0,0
3,86,640021012,3.0,1.0,NaN,NaN,NaN,0.100000,1,0,0,1,0,0
4,86,904026001,4.0,1.0,NaN,NaN,NaN,NaN,1,0,0,0,0,0


# Phase 5: The LightGBM Ranker:
The plan: a model that looks at each customer's 131 candidates and re-orders them, so the item they'll actually buy move into the top 12. To learn that, it needs examples where we know the answers: week 1,2,3.

The Training Data